In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import sys
sys.path.append("..")
from src.preprocessing import df_to_densities
from src.forecasting import cv


import warnings
from scipy.integrate import IntegrationWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=IntegrationWarning)

Steps: <br>
1. Use cross-validation to select the best parameters for in-sample KDE <br>
2. Use cross-validation to select the number of dimensions for the dFPC using the resulting parameters for KDE in 1.

In [2]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df = pd.read_excel(returns_path, index_col="time")

# 1. Selecting KDE parameters

In [3]:
# Parameters to cross-validate
density_param_grid_0 = [
    {'kernel': ['gaussian', 'epanechnikov'],
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}, 
    {'kernel': ['t_student'], 
        'df': range(2,6),
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}]

density_param_grid = list(ParameterGrid(density_param_grid_0))
len(density_param_grid)

24

In [4]:
params_dict = {}
records = []

normalize_options = [True, False]
total_params = len(density_param_grid) * len(normalize_options)
i=1
KdFPC_kwargs = {
    "lag_max": 5,
    "alpha": 0.10,
    "du": 0.05,
    "B": 1000,
    "p": 5,
    # "u": df_lqds_support,
    "select_ncomp": False,
    # "dimension": 2
}
for normalize in normalize_options:
    for d in range(2,7):
        KdFPC_kwargs.update(
            {"dimension": d}
            )
        for params in density_param_grid:
            print(f"({i}/{total_params}) | Normalize = {normalize} | {params}")
            i += 1

            df_support, df_densities = df_to_densities(
                                            df, 
                                            params, 
                                            normalize_densities=normalize,
                                            verbose=False)
            try:
                cv_measures = cv(df_densities, df_support, initial_window=100)
            except Exception as e:
                print(f"\t ERROR: {e}")
                continue
            for m in cv_measures:
                record = {
                    # density
                    "normalized_density": normalize,
                    
                    # parameters
                    "kernel": params["kernel"],
                    "bandwidth": params["bandwidth"],
                    "adaptive": params["adaptive"],

                    # CV info
                    "fold": m["fold"]+1,
                    "method": m["method"],

                    # metrics
                    "dFPC_dimensions":   d,
                    "KLD": float(m["KLD"]),
                    "JSD": float(m["JSD"]),
                    "L_1": float(m["L_1"]),
                    "L_2": float(m["L_2"]),
                    "L_INFTY": float(m["L_INFTY"]),
                }

                records.append(record)
            
results_df = pd.DataFrame(records)

(1/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'gaussian'}
	>>> cv 1/149
	 ERROR: Array must not contain infs or NaNs
(2/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'epanechnikov'}
	>>> cv 1/149
	 ERROR: Array must not contain infs or NaNs
(3/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'scott', 'kernel': 'gaussian'}
	>>> cv 1/149
	>>> cv 2/149
	>>> cv 3/149
	>>> cv 4/149
	>>> cv 5/149
	>>> cv 6/149
	>>> cv 7/149
	>>> cv 8/149
	>>> cv 9/149
	>>> cv 10/149
	>>> cv 11/149
	>>> cv 12/149
	>>> cv 13/149
	>>> cv 14/149
	>>> cv 15/149
	>>> cv 16/149
	>>> cv 17/149
	>>> cv 18/149
	>>> cv 19/149
	>>> cv 20/149
	>>> cv 21/149
	>>> cv 22/149
	>>> cv 23/149
	>>> cv 24/149
	>>> cv 25/149
	>>> cv 26/149
	>>> cv 27/149
	>>> cv 28/149
	>>> cv 29/149
	>>> cv 30/149
	>>> cv 31/149
	>>> cv 32/149
	>>> cv 33/149
	>>> cv 34/149
	>>> cv 35/149
	>>> cv 36/149
	>>> cv 37/149
	>>> cv 38/149
	>>> cv 39/149
	>>> cv 40/149
	>>> c

In [18]:
results_df.to_excel("../data/processed/cv_density_estimation_v2.xlsx", index=False)

In [27]:
# RESULTS BY MODEL
metrics = ["KLD", "JSD", "L_1", "L_2", "L_INFTY"]

mean_df = (
    results_df
    .groupby(['normalized_density', 'kernel', 'bandwidth',	'adaptive',	'method'])[metrics]
    .mean()
    .reset_index()
)
mean_df

,normalized_density,kernel,bandwidth,adaptive,method,KLD,JSD,L_1,L_2,L_INFTY
0,False,epanechnikov,scott,False,KLE,0.564823,0.010489,32878.650277,1760.368562,209.094657
1,False,epanechnikov,scott,True,KLE,0.229213,0.008642,31566.961246,1832.049633,249.345988
2,False,gaussian,scott,True,KLE,0.128304,0.005878,25448.966773,1363.193503,157.155147
3,False,t_student,scott,False,KLE,0.136366,0.005914,23196.732773,1132.757191,118.313098
4,False,t_student,scott,True,KLE,0.086657,0.005219,22094.973669,1198.394264,145.108980
5,False,t_student,silverman,False,KLE,0.225297,0.008554,28815.590965,1536.645414,201.274973
6,False,t_student,silverman,True,KLE,0.132109,0.007325,26917.456066,1654.699385,277.771804
7,True,epanechnikov,scott,False,KLE,0.094531,0.003846,19848.571186,1449.514707,205.825519
8,True,epanechnikov,scott,True,KLE,0.061142,0.004331,21160.892721,1684.452932,261.591930
9,True,gaussian,scott,True,KLE,0.036903,0.002505,15318.026892,1090.001149,143.604863


In [35]:
# BEST METHOD
for m in metrics:
    best_value = mean_df[m].min()
    mean_df[f"win_{m}"] = mean_df[m] == best_value

win_cols = [f"win_{m}" for m in metrics]

mean_df["n_wins"] = mean_df[win_cols].sum(axis=1)

best_overall = mean_df.sort_values("n_wins", ascending=False)

print(best_overall.head(3))

print('n')
print("Best method:", best_overall.iloc[0])

    normalized_density        kernel bandwidth  adaptive method       KLD  \
10                True     t_student     scott     False    KLE  0.033641   
11                True     t_student     scott      True    KLE  0.033303   
0                False  epanechnikov     scott     False    KLE  0.564823   

         JSD           L_1          L_2     L_INFTY  win_KLD  win_JSD  \
10  0.002037  13435.537880   838.789978   99.724172    False     True   
11  0.002278  14686.849557  1013.175067  134.154399     True    False   
0   0.010489  32878.650277  1760.368562  209.094657    False    False   

    win_L_1  win_L_2  win_L_INFTY  n_wins  
10     True     True         True       4  
11    False    False        False       1  
0     False    False        False       0  
n
Best method: normalized_density           True
kernel                  t_student
bandwidth                   scott
adaptive                    False
method                        KLE
KLD                      0.033641
JSD

In [36]:
# WINNER BY COUNT OF MIN ERROR BY FOLD
winners = (
    results_df
    .loc[results_df.groupby("fold")["JSD"].idxmin()]
    .value_counts(['normalized_density', 'kernel', 'bandwidth',	'adaptive',	'method'])
)

winners.reset_index()

,normalized_density,kernel,bandwidth,adaptive,method,count
0,True,t_student,scott,False,KLE,80
1,True,t_student,scott,True,KLE,23
2,True,t_student,silverman,False,KLE,19
3,True,t_student,silverman,True,KLE,12
4,True,gaussian,scott,True,KLE,10
5,True,epanechnikov,scott,False,KLE,2
6,True,epanechnikov,scott,True,KLE,1
7,False,t_student,scott,False,KLE,1
8,False,t_student,scott,True,KLE,1


# Checking models with error